# EquipX device detector: train on Colab

Trains a YOLO nano detector for the 10 EquipX devices and exports TFLite for the Android `TfliteDetector`.

**Before running**
1. Runtime > Change runtime type > **T4 GPU**.
2. Left sidebar > key icon (**Secrets**) > add `ROBOFLOW_API_KEY` (value = your Roboflow key), turn **Notebook access** on. The key is never printed or saved to files.
3. (Optional but strongly recommended) Own lab photos: put them in Google Drive at `MyDrive/equipx/own_photos/<class_id>/*.jpg` (class ids are listed in the next cell), or zip that folder and upload it in the "Own photos" cell.
4. Runtime > **Run all**. Final cell downloads `equipx_export.zip` and copies it to `MyDrive/equipx/export/`.

In [ ]:
import subprocess, torch
print(subprocess.run(['nvidia-smi','-L'],capture_output=True,text=True).stdout or 'NO GPU')
assert torch.cuda.is_available(), 'No GPU: Runtime > Change runtime type > T4 GPU'

In [ ]:
!pip install -q ultralytics roboflow pyyaml pillow numpy

## Config and class mapping (written from `ml/classes.yaml`)

In [ ]:
%%writefile /content/classes.yaml
# EquipX detector classes (order == content/index.json == model output class index)
classes:
  - digital_multimeter
  - analog_multimeter
  - digital_clamp_meter
  - digital_oscilloscope
  - function_generator
  - dc_power_supply
  - soldering_station
  - lcr_meter
  - insulation_tester
  - breadboard_trainer

# Public datasets (TASKS.md section 3). roboflow = workspace/project slugs.
# license_ok is from the Roboflow API metadata (checked 2026-11-05, see datasets_report.md).
# mapping: dataset class name -> our id. Matching is case-insensitive and ignores spaces/underscores/hyphens.
# Classes not listed are dropped (boxes removed; images left with no boxes are discarded).
datasets:
  tardata:
    roboflow: [new-workspace-febsz, tardata]
    license_ok: true
    mapping:
      Digital_Multimeter: digital_multimeter
      Analog_Multimeter: analog_multimeter
      Digital_Clamp_Meter: digital_clamp_meter
      Tektronik-1052B_Digital_Oscilloscope: digital_oscilloscope
      Tektronik-2000_Digital_Oscilloscope: digital_oscilloscope
      Function_Generator: function_generator
      Benchtop_Electrical_Power_Supply: dc_power_supply
      Power_Supply_30V2A: dc_power_supply
      Power_Supply_30V3A: dc_power_supply
      GORDAK-Soldering_Station: soldering_station
      HAKO-Soldering_Station: soldering_station
      IM3533-01_LCR_Meter: lcr_meter
      Insulation_Tester: insulation_tester
      Protoboard: breadboard_trainer
    # dropped on purpose: AC-Voltage&DC-Regulator_Power_Supply (AC regulator, not a DC bench supply), Soldering_iron
    # (bare iron, not a station), Digital_Power_Meter, Current_Probe, cables, hand tools, transformers
  objectdetection:
    roboflow: [universiti-malaysia-pahang-qcvas, objectdetection-ngxjp]
    license_ok: true
    mapping:
      "DC Power Supply": dc_power_supply
      "Digital Trainer": breadboard_trainer
      "Function Generator": function_generator
      "Digital Multimeter": digital_multimeter
      "Oscilloscope": digital_oscilloscope
  instrument_lab:
    roboflow: [universidad-de-los-andes-lzihh, instrument-lab-detection]
    license_ok: true
    mapping:
      fuente: dc_power_supply
      generador: function_generator
      multimetro: digital_multimeter
      osciloscopio: digital_oscilloscope
  equipment_data:
    roboflow: [senior-design-yolov7, equipment-data]
    license_ok: true   # API reports CC BY 4.0
    mapping:
      AFG: function_generator
      ELVIS: breadboard_trainer      # NI ELVIS = benchtop trainer workstation; review a few images
      "Digital Oscilliscope": digital_oscilloscope
      Multimeter: digital_multimeter
      "Power Supply": dc_power_supply
  testbw:
    roboflow: [new-workspace-febsz, testbw]
    license_ok: true
    mapping:
      Analog_Multimeter: analog_multimeter
      Digital_Clamp_Meter: digital_clamp_meter
      Digital_Multimeter: digital_multimeter
      Protoboard: breadboard_trainer
  resistor_detection:
    roboflow: [circuits-project, resistor-detection-5azes]
    license_ok: true
    mapping:
      breadboard: breadboard_trainer   # plain breadboard only; wire/resistor dropped
  # oscilloscope_detection (roboticslearning/oscilloscope-detection): CC BY 4.0 but 0 images and 0 versions
  # in the API (empty project), so it is not downloadable. Skipped.

In [ ]:
import os, yaml
os.makedirs('/content/datasets/raw', exist_ok=True)
CFG = yaml.safe_load(open('/content/classes.yaml'))
IDS = CFG['classes']
print(IDS)
EPOCHS, PATIENCE, IMGSZ = 60, 15, 640
WHOLE_IMAGE_BOXES = True   # own photos have no labels: one near-full-image box per photo (use only for close-up, single-device photos)

## Roboflow download (license-checked, YOLO format)
Only datasets with a CC BY / Public Domain / MIT license (from the API) are downloaded.

In [ ]:
import requests, shutil
from google.colab import userdata
from roboflow import Roboflow
try:
    KEY = userdata.get('ROBOFLOW_API_KEY')
except Exception as e:
    raise SystemExit('Add ROBOFLOW_API_KEY in Colab Secrets (key icon, left sidebar) and enable Notebook access, then re-run.')
rf = Roboflow(api_key=KEY)
OK_LICENSES = ('cc by', 'public domain', 'mit')   # CC BY 4.0, CC BY-SA, Public Domain, MIT
for name, d in CFG['datasets'].items():
    ws, proj = d['roboflow']
    meta = requests.get(f'https://api.roboflow.com/{ws}/{proj}', params={'api_key': KEY}, timeout=30).json()
    p, versions = meta['project'], meta.get('versions', [])
    lic = (p.get('license') or 'unknown')
    print(f"{name}: license={lic}, images={p.get('images')}, classes={list(p.get('classes', {}))}")
    if not any(lic.lower().startswith(l) for l in OK_LICENSES):
        print('   SKIP: license not allowed'); shutil.rmtree(f'/content/datasets/raw/{name}', ignore_errors=True); continue
    if not versions:
        print('   SKIP: no versions'); continue
    latest = max(versions, key=lambda v: v['created'])['id'].split('/')[-1]
    out = f'/content/datasets/raw/{name}'
    if os.path.exists(out + '/data.yaml'):
        print('   already downloaded'); continue
    rf.workspace(ws).project(proj).version(int(latest)).download('yolov8', location=out, overwrite=True)
    print(f'   downloaded version {latest} -> {out}')

## Own photos (optional)
Option A: Google Drive folder. Option B: upload a zip containing `<class_id>/*.jpg`.

In [ ]:
OWN = '/content/own_photos'
os.makedirs(OWN, exist_ok=True)
USE_DRIVE, UPLOAD_ZIP = True, False
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    src = '/content/drive/MyDrive/equipx/own_photos'
    if os.path.isdir(src):
        shutil.copytree(src, OWN, dirs_exist_ok=True)
    else:
        print('No Drive folder at', src, '(continuing with public data only)')
if UPLOAD_ZIP:
    from google.colab import files
    up = files.upload()
    for fn in up:
        shutil.unpack_archive(fn, OWN)
print({d: len(os.listdir(f'{OWN}/{d}')) for d in sorted(os.listdir(OWN)) if os.path.isdir(f'{OWN}/{d}')})

## Merge, remap, dedupe, split
Same code as `ml/merge.py`.

In [ ]:
%%writefile /content/merge.py
#!/usr/bin/env python3
"""Remap / dedupe / split Roboflow YOLO exports into one 10-class EquipX dataset.

Usage (works locally and in Colab):
  python merge.py --raw datasets/raw --out datasets/equipx --own own_photos [--whole-image-boxes]
Needs only: pyyaml, pillow, numpy.
"""
import argparse, random, re, shutil, sys
from collections import defaultdict
from pathlib import Path

import numpy as np
import yaml
from PIL import Image

IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}


def norm(s):
    return re.sub(r"[\s_\-]+", "", str(s)).lower()


def dhash(path, size=8):
    with Image.open(path) as im:
        g = im.convert("L").resize((size + 1, size), Image.BILINEAR)
    a = np.asarray(g, dtype=np.int16)
    bits = (a[:, 1:] > a[:, :-1]).flatten()
    return int("".join("1" if b else "0" for b in bits), 2)


def parse_label(path, cmap):
    """Return [(new_id, cx, cy, w, h)] for mapped classes. cmap: old_id -> new_id. Handles polygons."""
    out = []
    if not path.exists():
        return out
    for line in path.read_text().splitlines():
        p = line.split()
        if len(p) < 5:
            continue
        try:
            old = int(float(p[0]))
            vals = [float(v) for v in p[1:]]
        except ValueError:
            continue
        if old not in cmap:
            continue
        if len(vals) == 4:
            cx, cy, w, h = vals
        else:  # polygon -> bbox
            xs, ys = vals[0::2], vals[1::2]
            x0, x1, y0, y1 = min(xs), max(xs), min(ys), max(ys)
            cx, cy, w, h = (x0 + x1) / 2, (y0 + y1) / 2, x1 - x0, y1 - y0
        if w <= 0 or h <= 0:
            continue
        out.append((cmap[old], cx, cy, w, h))
    return out


def collect_public(raw, cfg, ids):
    items = []  # dict(src, boxes, group, origin)
    stats = {}
    for ds_dir in sorted(p for p in raw.iterdir() if p.is_dir()):
        name = ds_dir.name
        dcfg = cfg["datasets"].get(name)
        if not dcfg:
            print(f"[skip] {name}: not in classes.yaml")
            continue
        data = yaml.safe_load((ds_dir / "data.yaml").read_text())
        names = data["names"]
        names = [names[k] for k in sorted(names)] if isinstance(names, dict) else list(names)
        lut = {norm(k): v for k, v in dcfg["mapping"].items()}
        cmap, unmapped = {}, []
        for i, n in enumerate(names):
            tgt = lut.get(norm(n))
            if tgt:
                cmap[i] = ids.index(tgt)
            else:
                unmapped.append(n)
        print(f"[{name}] mapped {len(cmap)}/{len(names)}; dropped classes: {unmapped}")
        n_img = n_kept = 0
        for split in ("train", "valid", "val", "test"):
            idir = ds_dir / split / "images"
            if not idir.exists():
                continue
            for img in idir.iterdir():
                if img.suffix.lower() not in IMG_EXT:
                    continue
                n_img += 1
                boxes = parse_label(ds_dir / split / "labels" / (img.stem + ".txt"), cmap)
                if not boxes:
                    continue
                n_kept += 1
                stem = img.stem.split(".rf.")[0]  # augmented copies share the source stem
                items.append(dict(src=img, boxes=boxes, group=f"{name}:{stem}", origin=name, own=False))
        stats[name] = (n_img, n_kept)
    return items, stats


def collect_own(own, ids, whole):
    items, skipped = [], 0
    if not own.exists():
        return items
    for cdir in sorted(p for p in own.iterdir() if p.is_dir()):
        if cdir.name not in ids:
            print(f"[own] ignoring folder {cdir.name} (not a class id)")
            continue
        cid = ids.index(cdir.name)
        for img in sorted(cdir.iterdir()):
            if img.suffix.lower() not in IMG_EXT:
                continue
            txt = img.with_suffix(".txt")
            if txt.exists():
                boxes = [(cid, *b[1:]) for b in parse_label(txt, {k: k for k in range(len(ids))})]
            elif whole:
                boxes = [(cid, 0.5, 0.5, 0.98, 0.98)]
            else:
                boxes = []
            if not boxes:
                skipped += 1
                continue
            items.append(dict(src=img, boxes=boxes, group=f"own:{cdir.name}/{img.stem}", origin="own", own=True))
    if skipped:
        print(f"[own] {skipped} unlabeled photos skipped (add .txt labels or pass --whole-image-boxes)")
    return items


def dedupe(items, thresh=2):
    kept, hashes, dups = [], [], 0
    for it in items:
        try:
            h = dhash(it["src"])
        except Exception:
            continue
        if any(bin(h ^ o).count("1") <= thresh for o in hashes):
            dups += 1
            continue
        hashes.append(h)
        kept.append(it)
    print(f"[dedupe] removed {dups} near-duplicates, kept {len(kept)}")
    return kept


def split(items, ids, seed, own_test=0.6):
    rng = random.Random(seed)
    counts = defaultdict(int)
    for it in items:
        for b in it["boxes"]:
            counts[b[0]] += 1
    groups = defaultdict(list)
    for it in items:
        groups[it["group"]].append(it)
    by_primary = defaultdict(list)
    for g, its in groups.items():
        cls = {b[0] for it in its for b in it["boxes"]}
        by_primary[min(cls, key=lambda c: counts[c])].append(its)  # rarest class drives stratification
    out = {"train": [], "valid": [], "test": []}
    for cls, gl in by_primary.items():
        rng.shuffle(gl)
        own_g = [g for g in gl if g[0]["own"]]
        pub_g = [g for g in gl if not g[0]["own"]]
        for lst, (a, b) in ((pub_g, (0.8, 0.9)), (own_g, (1 - own_test - 0.2, 1 - own_test))):
            n = len(lst)
            i, j = round(n * a), round(n * b)
            for k, g in enumerate(lst):
                out["train" if k < i else "valid" if k < j else "test"].extend(g)
    return out


def write(out_dir, splits, ids):
    if out_dir.exists():
        shutil.rmtree(out_dir)
    for s, its in splits.items():
        (out_dir / s / "images").mkdir(parents=True)
        (out_dir / s / "labels").mkdir(parents=True)
        for n, it in enumerate(its):
            stem = f"{it['origin']}_{n:05d}"
            shutil.copy(it["src"], out_dir / s / "images" / (stem + it["src"].suffix.lower()))
            lines = [f"{c} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}" for c, cx, cy, w, h in it["boxes"]]
            (out_dir / s / "labels" / (stem + ".txt")).write_text("\n".join(lines) + "\n")
    (out_dir / "data.yaml").write_text(yaml.safe_dump(
        {"path": str(out_dir.resolve()), "train": "train/images", "val": "valid/images",
         "test": "test/images", "nc": len(ids), "names": {i: n for i, n in enumerate(ids)}}, sort_keys=False))


def report(splits, ids):
    print(f"\n{'class':22}" + "".join(f"{s+' img/box':>16}" for s in splits) + f"{'total img':>11}")
    weak = []
    for ci, name in enumerate(ids):
        row, tot = "", 0
        for s, its in splits.items():
            imgs = [it for it in its if any(b[0] == ci for b in it["boxes"])]
            boxes = sum(1 for it in its for b in it["boxes"] if b[0] == ci)
            row += f"{len(imgs):>9}/{boxes:<6}"
            tot += len(imgs)
        print(f"{name:22}{row}{tot:>11}")
        if tot < 100:
            weak.append((name, tot))
    print(f"{'ALL':22}" + "".join(f"{len(its):>16}" for its in splits.values()))
    for n, t in weak:
        print(f"WARNING: class '{n}' has only {t} images (< 100): add own photos in own_photos/{n}/")
    own_test = {c: 0 for c in ids}
    for it in splits["test"]:
        if it["own"]:
            own_test[ids[it["boxes"][0][0]]] += 1
    if not any(own_test.values()):
        print("NOTE: no own photos in the test split: metrics only reflect public data, not your lab.")


def main():
    here = Path(__file__).parent
    ap = argparse.ArgumentParser()
    ap.add_argument("--raw", default=str(here / "datasets/raw"))
    ap.add_argument("--out", default=str(here / "datasets/equipx"))
    ap.add_argument("--own", default=str(here / "own_photos"))
    ap.add_argument("--classes", default=str(here / "classes.yaml"))
    ap.add_argument("--whole-image-boxes", action="store_true",
                    help="unlabeled own photos get one near-full-image box of their folder's class")
    ap.add_argument("--own-test-fraction", type=float, default=0.6)
    ap.add_argument("--seed", type=int, default=0)
    a = ap.parse_args()
    cfg = yaml.safe_load(open(a.classes))
    ids = cfg["classes"]
    raw = Path(a.raw)
    if not raw.exists():
        sys.exit(f"{raw} not found: download datasets first")
    pub, stats = collect_public(raw, cfg, ids)
    own = collect_own(Path(a.own), ids, a.whole_image_boxes)
    print(f"public images with mapped boxes: {len(pub)}; own photos: {len(own)}")
    items = dedupe(pub) + own  # never dedupe away own photos
    splits = split(items, ids, a.seed, a.own_test_fraction)
    write(Path(a.out), splits, ids)
    report(splits, ids)
    print(f"\nwrote {a.out}/data.yaml")


if __name__ == "__main__":
    main()

In [ ]:
flag = '--whole-image-boxes' if WHOLE_IMAGE_BOXES else ''
!python /content/merge.py --raw /content/datasets/raw --out /content/datasets/equipx --own /content/own_photos --classes /content/classes.yaml {flag}

## Train (YOLO nano, CUDA)

In [ ]:
from ultralytics import YOLO
model = YOLO('yolo11n.pt')
model.train(data='/content/datasets/equipx/data.yaml', epochs=EPOCHS, patience=PATIENCE, imgsz=IMGSZ,
            batch=16, device=0, project='/content/runs', name='equipx', exist_ok=True, plots=True)
# If the session disconnects: YOLO('/content/runs/equipx/weights/last.pt').train(resume=True)

## Per-class metrics (test split)

In [ ]:
import json, pandas as pd
best = YOLO('/content/runs/equipx/weights/best.pt')
split = 'test' if os.listdir('/content/datasets/equipx/test/images') else 'val'
r = best.val(data='/content/datasets/equipx/data.yaml', split=split, imgsz=IMGSZ, device=0, plots=False)
rows = []
for k, ci in enumerate(r.box.ap_class_index):
    p, rec, ap50, ap = r.box.class_result(k)
    rows.append(dict(cls=IDS[ci], precision=round(p,3), recall=round(rec,3), mAP50=round(ap50,3), mAP50_95=round(ap,3)))
df = pd.DataFrame(rows)
missing = [c for c in IDS if c not in set(df.cls)]
display(df)
print('mAP50 overall:', round(float(r.box.map50),3), '| split:', split, '| classes with no test instances:', missing)
os.makedirs('/content/export', exist_ok=True)
json.dump(dict(split=split, overall=dict(precision=float(r.box.mp), recall=float(r.box.mr), mAP50=float(r.box.map50), mAP50_95=float(r.box.map)),
               per_class=rows, no_test_instances=missing), open('/content/export/metrics.json','w'), indent=2)

How to read: **precision** = share of detections that are right, **recall** = share of real devices found, **mAP50** = box quality at IoU 0.5.
Aim for mAP50 > 0.8 per class on your own-photo test split. Classes listed under "no test instances" or with very low recall need more own photos.

## Export TFLite (float16, raw output without NMS)

In [ ]:
import glob, shutil
best = YOLO('/content/runs/equipx/weights/best.pt')
best.export(format='tflite', imgsz=IMGSZ, half=True, nms=False)
f16 = sorted(glob.glob('/content/runs/equipx/weights/best_saved_model/*float16.tflite'))
print(f16)
shutil.copy(f16[0], '/content/export/equipx_detector.tflite')

In [ ]:
# Optional int8 (smaller/faster, small accuracy loss). Uses a slice of the dataset for calibration. Failure is OK: float16 is the default.
try:
    YOLO('/content/runs/equipx/weights/best.pt').export(format='tflite', imgsz=IMGSZ, int8=True, data='/content/datasets/equipx/data.yaml', fraction=0.2, nms=False)
    i8 = sorted(glob.glob('/content/runs/equipx/weights/best_saved_model/*int8.tflite'))
    shutil.copy(i8[0], '/content/export/equipx_detector_int8.tflite'); print('int8 OK', i8[0])
except Exception as e:
    print('int8 export failed (ignore, use float16):', repr(e)[:300])

## labels.txt + model_info.json (inspects the real tensors)

In [ ]:
import numpy as np, tensorflow as tf
from PIL import Image
open('/content/export/labels.txt','w').write('\n'.join(IDS) + '\n')
def inspect(path):
    it = tf.lite.Interpreter(model_path=path); it.allocate_tensors()
    i, o = it.get_input_details()[0], it.get_output_details()[0]
    img = sorted(glob.glob('/content/datasets/equipx/test/images/*') + glob.glob('/content/datasets/equipx/valid/images/*'))[0]
    x = np.asarray(Image.open(img).convert('RGB').resize((IMGSZ, IMGSZ)), dtype=np.float32)[None] / 255.0
    it.set_tensor(i['index'], x.astype(i['dtype']) if i['dtype'] == np.float32 else (x/i['quantization'][0] + i['quantization'][1]).astype(i['dtype']))
    it.invoke()
    y = it.get_tensor(o['index']).astype(np.float32)
    return it, i, o, y
it, i, o, y = inspect('/content/export/equipx_detector.tflite')
shape = [int(s) for s in o['shape']]
channels_first = shape[1] == 4 + len(IDS)
box_max = float(np.abs(y[0][:4] if channels_first else y[0][:, :4]).max())
info = dict(
    model='equipx_detector.tflite', task='detect', num_classes=len(IDS), classes=IDS, labels_file='labels.txt',
    input=dict(shape=[int(s) for s in i['shape']], layout='NHWC', dtype=str(np.dtype(i['dtype'])), color='RGB',
               normalization='pixel/255.0 -> [0,1]; resize/letterbox to square, no mean/std'),
    output=dict(shape=shape, dtype=str(np.dtype(o['dtype'])),
                layout='[1, 4+num_classes, num_anchors] channels-first: rows 0-3 = cx,cy,w,h; rows 4.. = class scores (already sigmoid, no objectness)' if channels_first
                       else '[1, num_anchors, 4+num_classes] anchors-first: cols 0-3 = cx,cy,w,h; cols 4.. = class scores (already sigmoid, no objectness)',
                box_format='cx,cy,w,h', box_units='normalized 0-1 of input size' if box_max <= 2 else 'pixels of input size (640)',
                nms='NOT applied: apply confidence threshold + class-wise NMS (IoU ~0.45) in app code'),
    export=dict(imgsz=IMGSZ, half=True, ultralytics=__import__('ultralytics').__version__),
)
json.dump(info, open('/content/export/model_info.json','w'), indent=2)
print(json.dumps(info, indent=2))

## Package and download

In [ ]:
for d in sorted(glob.glob('/content/runs/equipx/*.png'))[:4]: shutil.copy(d, '/content/export/')
shutil.copy('/content/datasets/equipx/data.yaml', '/content/export/data.yaml')
shutil.make_archive('/content/equipx_export', 'zip', '/content/export')
os.makedirs('/content/drive/MyDrive/equipx/export', exist_ok=True) if os.path.isdir('/content/drive/MyDrive') else None
if os.path.isdir('/content/drive/MyDrive/equipx'):
    shutil.copytree('/content/export', '/content/drive/MyDrive/equipx/export', dirs_exist_ok=True)
    shutil.copy('/content/runs/equipx/weights/best.pt', '/content/drive/MyDrive/equipx/export/best.pt')
    print('Saved a copy to Drive: MyDrive/equipx/export/')
from google.colab import files
files.download('/content/equipx_export.zip')